# CoT vs Chain_Code: Comparative Analysis (HML)

**Comparison of regular (CoT) vs chain_code mode across:**
- 15 models (requested DeepSeek/Qwen/Llama/GPT-OSS/Gemma/Phi suite)
- 3 datasets: HumanEval, MBPP, LiveCodeBench

**Analysis Focus:**
- Same model, same dataset comparisons
- Entropy & loss differences between modes
- Head importance pattern changes
- Knockout behavior differences

**Key Questions:**
1. How does analyzing code-only tokens (chain_code) differ from full CoT?
2. Do the same heads matter for both modes?
3. Is one mode more robust to head ablation?

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

BASE_DIR = Path('../../results/hml')
REQUESTED_MODELS = [
    'Qwen--Qwen3-0.6B',
    'Qwen--Qwen3-4B',
    'Qwen--Qwen3-8B',
    'Qwen--Qwen3-14B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-7B',
    'deepseek-ai--DeepSeek-R1-Distill-Llama-8B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-14B',
    'deepseek-ai--DeepSeek-R1-0528-Qwen3-8B',
    'meta-llama--Llama-3.2-3B-Instruct',
    'openai--gpt-oss-20b',
    'google--gemma-4-E2B-it',
    'google--gemma-4-E4B-it',
    'google--gemma-4-26B-A4B-it',
    'microsoft--Phi-4-mini-reasoning',
]
MODEL_DIR_BY_REQUEST = {
    'Qwen--Qwen3-0.6B': 'Qwen3-0.6B',
    'Qwen--Qwen3-4B': 'Qwen3-4B',
    'Qwen--Qwen3-8B': 'Qwen3-8B',
    'Qwen--Qwen3-14B': 'Qwen3-14B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B': 'DeepSeek-R1-Distill-Qwen-1.5B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-7B': 'DeepSeek-R1-Distill-Qwen-7B',
    'deepseek-ai--DeepSeek-R1-Distill-Llama-8B': 'DeepSeek-R1-Distill-Llama-8B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-14B': 'DeepSeek-R1-Distill-Qwen-14B',
    'deepseek-ai--DeepSeek-R1-0528-Qwen3-8B': 'DeepSeek-R1-0528-Qwen3-8B',
    'meta-llama--Llama-3.2-3B-Instruct': 'Llama-3.2-3B-Instruct',
    'openai--gpt-oss-20b': 'gpt-oss-20b',
    'google--gemma-4-E2B-it': 'gemma-4-E2B-it',
    'google--gemma-4-E4B-it': 'gemma-4-E4B-it',
    'google--gemma-4-26B-A4B-it': 'gemma-4-26B-A4B-it',
    'microsoft--Phi-4-mini-reasoning': 'Phi-4-mini-reasoning',
}
MODELS = [MODEL_DIR_BY_REQUEST[m] for m in REQUESTED_MODELS]
MODEL_DISPLAY = {MODEL_DIR_BY_REQUEST[m]: m for m in REQUESTED_MODELS}
MODEL_SHORT = {
    'Qwen3-0.6B': 'Q3-0.6B',
    'Qwen3-4B': 'Q3-4B',
    'Qwen3-8B': 'Q3-8B',
    'Qwen3-14B': 'Q3-14B',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'DS-DQ1.5B',
    'DeepSeek-R1-Distill-Qwen-7B': 'DS-DQ7B',
    'DeepSeek-R1-Distill-Llama-8B': 'DS-DL8B',
    'DeepSeek-R1-Distill-Qwen-14B': 'DS-DQ14B',
    'DeepSeek-R1-0528-Qwen3-8B': 'DS-0528-Q8B',
    'Llama-3.2-3B-Instruct': 'Llama3.2-3B',
    'gpt-oss-20b': 'gpt-oss-20b',
    'gemma-4-E2B-it': 'Gemma4-E2B',
    'gemma-4-E4B-it': 'Gemma4-E4B',
    'gemma-4-26B-A4B-it': 'Gemma4-26B',
    'Phi-4-mini-reasoning': 'Phi4-mini',
}
DATASETS = ['humaneval', 'mbpp', 'livecodebench']
MODES = ['CoT', 'Chain_Code']

palette = sns.color_palette('tab20', n_colors=max(12, len(MODELS)))
MODEL_COLORS = {model: palette[i] for i, model in enumerate(MODELS)}
DS_COLORS = {'humaneval': 'tab:green', 'mbpp': 'tab:purple', 'livecodebench': 'tab:red'}
MODE_COLORS = {'CoT': 'tab:blue', 'Chain_Code': 'tab:red'}
MODE_STYLES = {'CoT': '-', 'Chain_Code': '--'}


def model_label(model, short=False):
    if short:
        return MODEL_SHORT.get(model, model)
    return MODEL_DISPLAY.get(model, model)


def symmetric_percentile_limits(arrays, low=1, high=99):
    values = []
    for arr in arrays:
        if arr is None:
            continue
        arr = np.asarray(arr)
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            continue
        values.append(np.percentile(finite, low))
        values.append(np.percentile(finite, high))

    if not values:
        return -1.0, 1.0

    max_abs_val = max(abs(min(values)), abs(max(values)))
    if not np.isfinite(max_abs_val) or max_abs_val == 0:
        max_abs_val = 1.0
    return -max_abs_val, max_abs_val

## 1. Data Loading

### 1.1 Load Entropy & Loss Data

In [ ]:
def load_entropy_data(dataset, model, mode='CoT'):
    """Load per_token.npz for entropy/loss data."""
    subdir = 'entropy_loss_results' if mode == 'CoT' else 'entropy_loss_results_chain_code'
    path = BASE_DIR / dataset / subdir / model / dataset / 'per_token.npz'
    if not path.exists():
        return None
    data = np.load(path)
    entropy_keys = sorted([k for k in data.keys() if k.startswith('entropy_layer_')],
                          key=lambda k: int(k.split('_')[-1]))
    loss_keys = sorted([k for k in data.keys() if k.startswith('loss_layer_')],
                       key=lambda k: int(k.split('_')[-1]))
    return {
        'avg_entropy': np.array([np.nanmean(data[k]) for k in entropy_keys]),
        'std_entropy': np.array([np.nanstd(data[k]) for k in entropy_keys]),
        'avg_loss': np.array([np.nanmean(data[k]) for k in loss_keys]),
        'std_loss': np.array([np.nanstd(data[k]) for k in loss_keys]),
        'num_layers': len(entropy_keys),
    }

entropy_data = {mode: {} for mode in MODES}
for mode in MODES:
    for model in MODELS:
        entropy_data[mode][model] = {}
        for ds in DATASETS:
            data = load_entropy_data(ds, model, mode)
            if data:
                entropy_data[mode][model][ds] = data

# Print summary
for mode in MODES:
    print(f'=== {mode} ===')
    for model in MODELS:
        for ds in DATASETS:
            d = entropy_data[mode][model].get(ds)
            if d:
                print(f'  {model}/{ds}: {d["num_layers"]}L, final_entropy={d["avg_entropy"][-1]:.4f}, final_loss={d["avg_loss"][-1]:.4f}')
            else:
                print(f'  {model}/{ds}: NOT FOUND')
    print()

### 1.2 Load Head Ablation Data

In [ ]:
def load_ablation_data(dataset, model, mode='CoT'):
    """Load ablation.npz and knockout.npz."""
    subdir = 'head_ablation_results' if mode == 'CoT' else 'head_ablation_results_chain_code'
    ablation_path = BASE_DIR / dataset / subdir / model / dataset / 'ablation.npz'
    knockout_path = BASE_DIR / dataset / subdir / model / dataset / 'knockout.npz'
    if not ablation_path.exists():
        return None
    ablation = np.load(ablation_path)
    result = {
        'ablation_mean': ablation['ablation_mean'],
        'num_layers': ablation['ablation_mean'].shape[0],
        'num_heads': ablation['ablation_mean'].shape[1],
    }
    if knockout_path.exists():
        ko = np.load(knockout_path)
        key = 'knockout_losses' if 'knockout_losses' in ko else 'avg_losses'
        if key in ko:
            result['knockout_losses'] = ko[key]
    return result

ablation_data = {mode: {} for mode in MODES}
for mode in MODES:
    for model in MODELS:
        ablation_data[mode][model] = {}
        for ds in DATASETS:
            data = load_ablation_data(ds, model, mode)
            if data:
                ablation_data[mode][model][ds] = data

for mode in MODES:
    print(f'=== {mode} (ablation) ===')
    for model in MODELS:
        for ds in DATASETS:
            d = ablation_data[mode][model].get(ds)
            if d:
                print(f'  {model}/{ds}: {d["num_layers"]}L x {d["num_heads"]}H')
            else:
                print(f'  {model}/{ds}: NOT FOUND')
    print()

## 2. Entropy & Loss Comparison

### 2.1 Final Layer Comparison

In [ ]:
comparison_data = []
for model in MODELS:
    for ds in DATASETS:
        cot = entropy_data['CoT'][model].get(ds)
        cc = entropy_data['Chain_Code'][model].get(ds)
        if cot and cc:
            comparison_data.append({
                'Model': model, 'Dataset': ds,
                'CoT_Entropy': cot['avg_entropy'][-1],
                'CC_Entropy': cc['avg_entropy'][-1],
                'Delta_Entropy': cc['avg_entropy'][-1] - cot['avg_entropy'][-1],
                'CoT_Loss': cot['avg_loss'][-1],
                'CC_Loss': cc['avg_loss'][-1],
                'Delta_Loss': cc['avg_loss'][-1] - cot['avg_loss'][-1],
            })

comparison_df = pd.DataFrame(comparison_data)
if not comparison_df.empty:
    print(comparison_df.to_string(index=False, float_format='%.4f'))

### 2.2 Delta Bar Charts

In [ ]:
if not comparison_df.empty:
    n_cols = min(4, len(MODELS))
    n_rows = int(np.ceil(len(MODELS) / n_cols))

    metric_specs = [
        ('Delta_Entropy', 'Delta Entropy (CC - CoT)', 'Entropy Delta by Model'),
        ('Delta_Loss', 'Delta Loss (CC - CoT)', 'Loss Delta by Model'),
    ]

    for metric, ylabel, title in metric_specs:
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 4.2 * n_rows), squeeze=False)

        for m_idx, model in enumerate(MODELS):
            r, c = divmod(m_idx, n_cols)
            ax = axes[r, c]
            mdf = comparison_df[comparison_df['Model'] == model]

            if mdf.empty:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center')
                ax.set_axis_off()
                continue

            x = np.arange(len(mdf))
            ax.bar(x, mdf[metric], color=[DS_COLORS[d] for d in mdf['Dataset']])
            ax.set_xticks(x)
            ax.set_xticklabels(mdf['Dataset'], rotation=20, ha='right')
            ax.axhline(0, color='black', linewidth=0.7)
            ax.set_ylabel(ylabel)
            ax.set_title(model_label(model, short=True), fontsize=11)

        total_slots = n_rows * n_cols
        for empty_idx in range(len(MODELS), total_slots):
            r, c = divmod(empty_idx, n_cols)
            axes[r, c].axis('off')

        fig.suptitle(title, fontsize=15, y=1.01)
        fig.tight_layout()
        plt.show()

### 2.3 Layer-wise Comparison

In [ ]:
for model in MODELS:
    fig, axes = plt.subplots(len(DATASETS), 2, figsize=(18, 5*len(DATASETS)), squeeze=False)
    for row, ds in enumerate(DATASETS):
        ax_ent, ax_loss = axes[row]
        for mode in MODES:
            data = entropy_data[mode][model].get(ds)
            if data is None:
                continue
            x = np.arange(1, data['num_layers']+1)
            ax_ent.plot(x, data['avg_entropy'], MODE_STYLES[mode], color=MODE_COLORS[mode],
                       linewidth=2, label=mode)
            ax_ent.fill_between(x, data['avg_entropy']-data['std_entropy'],
                               data['avg_entropy']+data['std_entropy'],
                               alpha=0.1, color=MODE_COLORS[mode])
            ax_loss.plot(x, data['avg_loss'], MODE_STYLES[mode], color=MODE_COLORS[mode],
                        linewidth=2, label=mode)
            ax_loss.fill_between(x, data['avg_loss']-data['std_loss'],
                                data['avg_loss']+data['std_loss'],
                                alpha=0.1, color=MODE_COLORS[mode])
        ax_ent.set_xlabel('Layer'); ax_ent.set_ylabel('Entropy')
        ax_ent.set_title(f'{ds} - Entropy', fontsize=12); ax_ent.legend()
        ax_loss.set_xlabel('Layer'); ax_loss.set_ylabel('Loss')
        ax_loss.set_title(f'{ds} - Loss', fontsize=12); ax_loss.legend()
    fig.suptitle(f'{model} - Layer-wise CoT vs Chain_Code', fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()

### 2.4 Statistical Summary

In [ ]:
print('\n' + '='*80)
print('ENTROPY & LOSS: CoT vs Chain_Code')
print('='*80)

for model in MODELS:
    mdf = comparison_df[comparison_df['Model'] == model]
    if mdf.empty:
        continue
    entropy_cc_lower = (mdf['Delta_Entropy'] < 0).sum()
    loss_cc_lower = (mdf['Delta_Loss'] < 0).sum()
    total = len(mdf)
    print(f'\n{model}:')
    print(f'  Chain_Code has LOWER entropy in {entropy_cc_lower}/{total} datasets')
    print(f'  Chain_Code has LOWER loss    in {loss_cc_lower}/{total} datasets')
    print(f'  Avg entropy delta: {mdf["Delta_Entropy"].mean():+.4f}')
    print(f'  Avg loss delta:    {mdf["Delta_Loss"].mean():+.4f}')

## 3. Head Ablation Comparison

### 3.1 Head Importance Statistics

In [ ]:
ablation_comparison = []
for model in MODELS:
    for ds in DATASETS:
        cot = ablation_data['CoT'][model].get(ds)
        cc = ablation_data['Chain_Code'][model].get(ds)
        if cot and cc:
            cot_flat = cot['ablation_mean'].flatten()
            cc_flat = cc['ablation_mean'].flatten()
            ablation_comparison.append({
                'Model': model, 'Dataset': ds,
                'CoT_Max': cot_flat.max(), 'CC_Max': cc_flat.max(),
                'Delta_Max': cc_flat.max() - cot_flat.max(),
                'CoT_Mean': cot_flat.mean(), 'CC_Mean': cc_flat.mean(),
                'Delta_Mean': cc_flat.mean() - cot_flat.mean(),
            })

ablation_comp_df = pd.DataFrame(ablation_comparison)
if not ablation_comp_df.empty:
    print(ablation_comp_df.to_string(index=False, float_format='%.4f'))

### 3.2 Top Head Consistency Across Modes

In [ ]:
K_COMPARE = 10

def get_top_k_heads(data, k=10):
    if data is None:
        return set()
    flat = data['ablation_mean'].flatten()
    top_k_idx = np.argsort(flat)[-k:][::-1]
    nh = data['num_heads']
    return {(idx // nh, idx % nh) for idx in top_k_idx}

print(f'Top-{K_COMPARE} Head Jaccard Similarity (CoT vs Chain_Code):\n')
print(f'{"Model":>25} | {"Dataset":>13} | {"Jaccard":>8} | {"Overlap":>8}')
print('-'*65)

for model in MODELS:
    for ds in DATASETS:
        cot_heads = get_top_k_heads(ablation_data['CoT'][model].get(ds), K_COMPARE)
        cc_heads = get_top_k_heads(ablation_data['Chain_Code'][model].get(ds), K_COMPARE)
        if cot_heads and cc_heads:
            inter = len(cot_heads & cc_heads)
            union = len(cot_heads | cc_heads)
            jaccard = inter / union if union > 0 else 0
            print(f'{model:>25} | {ds:>13} | {jaccard:8.3f} | {inter:>4}/{K_COMPARE}')

### 3.3 Side-by-Side Heatmaps

In [ ]:
for model in MODELS:
    heatmap_arrays = []
    for ds in DATASETS:
        cot = ablation_data['CoT'][model].get(ds)
        cc = ablation_data['Chain_Code'][model].get(ds)
        if cot is not None:
            heatmap_arrays.append(cot['ablation_mean'])
        if cc is not None:
            heatmap_arrays.append(cc['ablation_mean'])
        if cot is not None and cc is not None:
            heatmap_arrays.append(cc['ablation_mean'] - cot['ablation_mean'])

    symmetric_vmin, symmetric_vmax = symmetric_percentile_limits(heatmap_arrays)

    fig, axes = plt.subplots(len(DATASETS), 3, figsize=(21 * 0.95, 6 * len(DATASETS)), squeeze=False)
    mappable = None

    for row, ds in enumerate(DATASETS):
        cot = ablation_data['CoT'][model].get(ds)
        cc = ablation_data['Chain_Code'][model].get(ds)

        for col, (name, data) in enumerate([('CoT', cot), ('Chain_Code', cc)]):
            ax = axes[row, col]
            if data is None:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            else:
                mappable = ax.imshow(
                    data['ablation_mean'],
                    aspect='auto',
                    cmap='coolwarm',
                    vmin=symmetric_vmin,
                    vmax=symmetric_vmax,
                )
            ax.set_title(f'{ds} - {name}', fontsize=12)
            ax.set_xlabel('Head'); ax.set_ylabel('Layer')

        # Difference
        ax = axes[row, 2]
        if cot is not None and cc is not None:
            diff = cc['ablation_mean'] - cot['ablation_mean']
            mappable = ax.imshow(
                diff,
                aspect='auto',
                cmap='coolwarm',
                vmin=symmetric_vmin,
                vmax=symmetric_vmax,
            )
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        ax.set_title(f'{ds} - Diff (CC - CoT)', fontsize=12)
        ax.set_xlabel('Head'); ax.set_ylabel('Layer')

    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    if mappable is not None:
        fig.colorbar(mappable, cax=cbar_ax)

    fig.suptitle(f'{model} - Ablation: CoT vs Chain_Code', fontsize=15, y=1.02)
    plt.show()

### 3.4 Layer-wise Importance by Mode

In [ ]:
for model in MODELS:
    # Changed to 2 rows to accommodate the zoomed plots, increased figure height
    fig, axes = plt.subplots(2, len(DATASETS), figsize=(7*len(DATASETS), 12), squeeze=False)
    
    for d_idx, ds in enumerate(DATASETS):
        ax_orig = axes[0, d_idx]
        ax_zoom = axes[1, d_idx]
        
        skip_layers = 3 # Number of initial layers to drop for zoomed view
        
        for mode in MODES:
            data = ablation_data[mode][model].get(ds)
            if data is None:
                continue
            layer_avg = data['ablation_mean'].mean(axis=1)
            
            # --- Top Row: Original Plot ---
            ax_orig.plot(range(len(layer_avg)), layer_avg, MODE_STYLES[mode],
                         color=MODE_COLORS[mode], linewidth=2, label=mode)
            
            # --- Bottom Row: Zoomed Plot (only if Llama or DeepSeek) ---
            if ("Llama" in model or "DeepSeek" in model) and len(layer_avg) > skip_layers:
                ax_zoom.plot(range(skip_layers, len(layer_avg)), layer_avg[skip_layers:], 
                             MODE_STYLES[mode], color=MODE_COLORS[mode], linewidth=2, label=mode)

        # Format Original Axes
        ax_orig.set_xlabel('Layer')
        ax_orig.set_ylabel('Mean Importance')
        ax_orig.set_title(f'{ds}', fontsize=13)
        ax_orig.legend()
        ax_orig.grid(True, alpha=0.3)
        
        # Format Zoomed Axes or turn off if not needed
        if "Llama" in model or "DeepSeek" in model:
            ax_zoom.set_xlabel('Layer')
            ax_zoom.set_ylabel('Mean Importance')
            ax_zoom.set_title(f'{ds} (Zoomed: Layers {skip_layers}+)', fontsize=13)
            ax_zoom.legend()
            ax_zoom.grid(True, alpha=0.3)
        else:
            ax_zoom.axis('off')
            
    fig.suptitle(f'{model} - Layer-wise Importance', fontsize=15, y=0.98)
    fig.tight_layout()
    plt.show()

## 4. Knockout Comparison

### 4.1 Knockout Curves

In [ ]:
for model in MODELS:
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(7*len(DATASETS), 6), squeeze=False)
    for d_idx, ds in enumerate(DATASETS):
        ax = axes[0, d_idx]
        for mode in MODES:
            data = ablation_data[mode][model].get(ds)
            if data is None or 'knockout_losses' not in data:
                continue
            losses = data['knockout_losses']
            ax.plot(np.arange(len(losses)), losses, MODE_STYLES[mode],
                   color=MODE_COLORS[mode], linewidth=2, label=mode)
        ax.set_xlabel('Heads Knocked Out'); ax.set_ylabel('Loss')
        ax.set_title(f'{ds}', fontsize=13); ax.legend()
        ax.grid(True, alpha=0.3)
    fig.suptitle(f'{model} - Knockout Curves: CoT vs Chain_Code', fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()

### 4.2 Knockout Robustness

In [ ]:
knockout_comparison = []
for model in MODELS:
    for ds in DATASETS:
        cot = ablation_data['CoT'][model].get(ds)
        cc = ablation_data['Chain_Code'][model].get(ds)
        if not (cot and cc and 'knockout_losses' in cot and 'knockout_losses' in cc):
            continue
        cot_ko = cot['knockout_losses']
        cc_ko = cc['knockout_losses']
        # Normalize to % increase from baseline
        cot_norm = (cot_ko - cot_ko[0]) / cot_ko[0] * 100
        cc_norm = (cc_ko - cc_ko[0]) / cc_ko[0] * 100
        # AUC of normalized curve (first 20 knockouts)
        n = min(20, len(cot_norm), len(cc_norm))
        knockout_comparison.append({
            'Model': model, 'Dataset': ds,
            'CoT_AUC': np.trapezoid(cot_norm[:n]),
            'CC_AUC': np.trapezoid(cc_norm[:n]),
            'Delta_AUC': np.trapezoid(cc_norm[:n]) - np.trapezoid(cot_norm[:n]),
            'CoT_Final': cot_norm[n-1],
            'CC_Final': cc_norm[n-1],
        })

knockout_comp_df = pd.DataFrame(knockout_comparison)
if not knockout_comp_df.empty:
    print('Knockout robustness (first 20 heads):')
    print(knockout_comp_df.to_string(index=False, float_format='%.2f'))
    print('\nPositive Delta_AUC = Chain_Code degrades faster (less robust)')

## 5. Combined Analysis

### 5.1 Correlation: Entropy Change vs Head Importance Change

In [ ]:
if not comparison_df.empty and not ablation_comp_df.empty:
    merged = comparison_df.merge(
        ablation_comp_df[['Model', 'Dataset', 'Delta_Max', 'Delta_Mean']],
        on=['Model', 'Dataset'], how='inner'
    )
    if not knockout_comp_df.empty:
        merged = merged.merge(
            knockout_comp_df[['Model', 'Dataset', 'Delta_AUC']],
            on=['Model', 'Dataset'], how='left'
        )
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for model in MODELS:
        mdf = merged[merged['Model'] == model]
        if mdf.empty:
            continue
        color = MODEL_COLORS[model]
        label = model_label(model, short=True)
        axes[0].scatter(mdf['Delta_Entropy'], mdf['Delta_Max'], color=color,
                       s=100, label=label, zorder=3)
        for _, r in mdf.iterrows():
            axes[0].annotate(r['Dataset'][:3], (r['Delta_Entropy'], r['Delta_Max']),
                           fontsize=8, ha='left', va='bottom')
        axes[1].scatter(mdf['Delta_Loss'], mdf['Delta_Mean'], color=color,
                       s=100, label=label, zorder=3)
        for _, r in mdf.iterrows():
            axes[1].annotate(r['Dataset'][:3], (r['Delta_Loss'], r['Delta_Mean']),
                           fontsize=8, ha='left', va='bottom')
    axes[0].set_xlabel('Delta Entropy'); axes[0].set_ylabel('Delta Max Importance')
    axes[0].set_title('Entropy vs Max Importance Change')
    axes[0].axhline(0, color='gray', lw=0.5); axes[0].axvline(0, color='gray', lw=0.5)
    axes[1].set_xlabel('Delta Loss'); axes[1].set_ylabel('Delta Mean Importance')
    axes[1].set_title('Loss vs Mean Importance Change')
    axes[1].axhline(0, color='gray', lw=0.5); axes[1].axvline(0, color='gray', lw=0.5)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), title='Model')
    fig.tight_layout(rect=[0, 0, 0.84, 1])
    plt.show()

### 5.2 Comprehensive Heatmap

In [ ]:
if not comparison_df.empty and not ablation_comp_df.empty:
    merged = comparison_df.merge(
        ablation_comp_df[['Model', 'Dataset', 'Delta_Max', 'Delta_Mean']],
        on=['Model', 'Dataset'], how='inner'
    )
    model_order = {m: i for i, m in enumerate(MODELS)}
    merged['ModelOrder'] = merged['Model'].map(model_order)
    merged = merged.sort_values(['ModelOrder', 'Dataset'])
    merged['Label'] = merged['Model'].map(lambda m: MODEL_SHORT.get(m, m)) + ' / ' + merged['Dataset']

    metrics = ['Delta_Entropy', 'Delta_Loss', 'Delta_Max', 'Delta_Mean']
    available = [m for m in metrics if m in merged.columns]
    if 'Delta_AUC' in merged.columns:
        available.append('Delta_AUC')

    heatmap_data = merged.set_index('Label')[available]
    symmetric_vmin, symmetric_vmax = symmetric_percentile_limits([heatmap_data.values])

    fig, ax = plt.subplots(figsize=(11, max(5, len(heatmap_data) * 0.4)))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt='.3f',
        cmap='coolwarm',
        center=0,
        vmin=symmetric_vmin,
        vmax=symmetric_vmax,
        ax=ax,
        linewidths=0.5,
    )
    ax.set_title('All Deltas (Chain_Code - CoT)', fontsize=14)
    fig.tight_layout()
    plt.show()

## 6. Key Findings

In [ ]:
print('\n' + '='*80)
print('KEY FINDINGS: CoT vs Chain_Code (HML)')
print('='*80)

print('\n1. ENTROPY & LOSS:')
for model in MODELS:
    mdf = comparison_df[comparison_df['Model'] == model]
    if mdf.empty: continue
    print(f'\n  {model}:')
    for _, r in mdf.iterrows():
        print(f'    {r["Dataset"]:15s} entropy delta={r["Delta_Entropy"]:+.4f}  loss delta={r["Delta_Loss"]:+.4f}')

print('\n2. HEAD IMPORTANCE:')
for model in MODELS:
    mdf = ablation_comp_df[ablation_comp_df['Model'] == model]
    if mdf.empty: continue
    print(f'\n  {model}:')
    for _, r in mdf.iterrows():
        print(f'    {r["Dataset"]:15s} max_delta={r["Delta_Max"]:+.4f}  mean_delta={r["Delta_Mean"]:+.4f}')

if not knockout_comp_df.empty:
    print('\n3. KNOCKOUT ROBUSTNESS:')
    for model in MODELS:
        mdf = knockout_comp_df[knockout_comp_df['Model'] == model]
        if mdf.empty: continue
        print(f'\n  {model}:')
        for _, r in mdf.iterrows():
            print(f'    {r["Dataset"]:15s} AUC_delta={r["Delta_AUC"]:+.2f} (positive = CC less robust)')